# Writing Papers with Jupyter Notebooks and Quarto
A very convenient way to write academic papers which include the code for all reported analyses is to use Jupyter notebooks together with [Quarto](https://quarto.org/). Quarto is a scientific publishing system that allows you to combine text, code, and results in a single document. Quarto will render Jupyter notebooks to PDF using LaTeX. When you include tables created with MakeTables in your notebook, they will be rendered directly on the screen in the notebook (as HTML via [Great Tables](https://posit-dev.github.io/great-tables/articles/intro.html)) but will be rendered as LaTeX tables in the final PDF document by Quarto. You can include analyses using different packages, in particular [pyfixest](https://py-econometrics.github.io/pyfixest/pyfixest.html) or [statsmodels](https://www.statsmodels.org/stable/index.html) or even Stata.


# An Example Notebook

Below is a short example paper written as a Jupyter notebook. The first tab shows the complete notebook, cell by cell: the raw cell at the top holds the YAML header with Quarto's PDF settings, and the `#|` comments in the code cells give each table a label (for cross-references such as `@tbl-1`) and a caption. The second tab shows the PDF that Quarto renders from it.

- Download the notebook: [quartoExample.ipynb](examples/quartoExample.ipynb){download="quartoExample.ipynb"} ([view on GitHub](https://github.com/py-econometrics/maketables/blob/main/docs/examples/quartoExample.ipynb))
- Download the BibTeX file with the references: [references.bib](examples/references.bib){download="references.bib"}
- Download the rendered PDF: [quartoExample.pdf](examples/quartoExample.pdf){download="quartoExample.pdf"}

To create the PDF yourself, put the notebook and `references.bib` in the same folder and run `quarto render quartoExample.ipynb`. This needs [Quarto](https://quarto.org/docs/get-started/) with a LaTeX installation (e.g. `quarto install tinytex`), and the Python packages used in the notebook (`maketables`, `pyfixest`, `statsmodels`).

::: {.panel-tabset}

## Notebook source


In [1]:
#| echo: false
#| output: asis

# Show every cell of the example notebook as a code block.
import json
from pathlib import Path

from IPython.display import Markdown

nb = json.loads(Path("examples/quartoExample.ipynb").read_text(encoding="utf-8"))
kinds = {
    "raw": ("yaml", "Raw cell (YAML header)"),
    "markdown": ("markdown", "Markdown cell"),
    "code": ("python", "Code cell"),
}
blocks = []
for i, cell in enumerate(nb["cells"], 1):
    lang, kind = kinds[cell["cell_type"]]
    src = "".join(cell["source"]).strip()
    # Escape shortcodes such as {{< pagebreak >}} so Quarto shows them as text
    src = src.replace("{{<", "{{{<").replace(">}}", ">}}}")
    header = "````{." + lang + ' filename="Cell ' + str(i) + ": " + kind + '"}'
    blocks.append("\n".join([header, src, "````"]))
Markdown("\n\n".join(blocks))

````{.yaml filename="Cell 1: Raw cell (YAML header)"}
---
title: "Quarto and MakeTables"
format:
  pdf:
    keep-tex: true
    papersize: a4
    fontsize: 11pt
    fontfamily: newpxtext
    link-citations: true
    linkcolor: "black"
    citecolor: "black"
    number-sections: true
    indent: "0m"
    cite-method: citeproc
    include-in-header: 
      - text: |
          \usepackage{setspace}
          \usepackage{float}
          \usepackage{booktabs}
          \usepackage{makecell}
          \usepackage{threeparttable}
          \usepackage{tabularx}
          \usepackage{multirow}
          \usepackage{array}
execute:
  echo: false
  warning: false
  cache: false
freeze: false
bibliography: references.bib
csl: https://www.zotero.org/styles/apa
author:
  - Peter Pan^[University of Nowhere]
date: "May, 2025"
date-format: long
abstract: |
  This document illustrates how to use MakeTables to write academic papers. Just write your paper in markdown in Jupyter Notebooks. Tables are displayed as HTML interactively when you work in the notebook on the screen. When you render the notebook as PDF with Quarto, MakeTable generates LaTeX code used by Quarto.
  

  \newpage
---
````

````{.python filename="Cell 2: Code cell"}
#| echo: false
#| include: false
#| eval: true

# Import necessary libraries
import numpy as np
import pandas as pd
import pyfixest as pf
import maketables as mt
import statsmodels.formula.api as smf

# Load your data (here using a sample Stata dataset with the import_dta function that also stores variable labels)
df = mt.import_dta("https://www.stata-press.com/data/r18/auto.dta")
````

````{.markdown filename="Cell 3: Markdown cell"}
\onehalfspacing

## Introduction
A very convenient way to write academic papers which include the code for all reported analyses is to use Jupyter notebooks together with [Quarto](https://quarto.org/). Quarto is a scientific publishing system that allows you to combine text, code, and results in a single document. Quarto will render Jupyter notebooks to PDF using LaTeX. When you include tables created with MakeTables in your notebook, they will be rendered directly on the screen in the notebook (as HTML via [Great Tables](https://posit-dev.github.io/great-tables/articles/intro.html)) but will be rendered as LaTeX tables in the final PDF document by Quarto.

Here is simple example of how to use MakeTables in a notebook to write a paper. It generates tables, for instance based on regressions in either [pyfixest](https://py-econometrics.github.io/pyfixest/pyfixest.html) or [statsmodels](https://www.statsmodels.org/stable/index.html). Just render this notebook with `quarto render quartoExample.ipynb` to create the PDF document. 

Note it is also very easy to include references and citations in your Quarto document (see for instance @wooldridge2010econometric and @greene2018econometric).
````

````{.markdown filename="Cell 4: Markdown cell"}
## Some Tables

Table @tbl-1 shows some descriptive statistics.
````

````{.python filename="Cell 5: Code cell"}
#| label: tbl-1
#| tbl-cap: "Automotive Data Summary"
#| tbl-pos: "H"

# Create a descriptive statistics table 
# Note that we can also pass through output-specific styling parameters
# Here we for instance want the first column with the variable names im LaTeX output to be slightly larger
mt.DTable(
    df, 
    vars=['mpg', 'weight', 'price'],
    stats=['count', 'mean', 'std'], 
    bycol=['foreign'],
    tex_style={'first_col_width': '3cm'},
)
````

````{.markdown filename="Cell 6: Markdown cell"}
Table @tbl-2 shows regression results here estimated with the excellent `pyfixest` package.
````

````{.python filename="Cell 7: Code cell"}
#| label: tbl-2
#| tbl-cap: "A Regression Table (PyFixest)"
#| tbl-pos: "H"

# Fit models using pyfixest
est1 = pf.feols("mpg ~ weight", data=df)
est2 = pf.feols("mpg ~ weight + length", data=df)

# Make the table
mt.ETable([est1, est2])
````

````{.markdown filename="Cell 8: Markdown cell"}
Table @tbl-3 uses `statsmodels` instead to estimate a probit model.
````

````{.python filename="Cell 9: Code cell"}
#| label: tbl-3
#| tbl-cap: "A Regression Table (Statsmodels)"
#| tbl-pos: "H"
 
# Generate a dummy variable and label it
df["foreign_i"] = (df["foreign"] == "Foreign")*1
mt.set_var_labels(df, {"foreign_i": "Foreign (indicator)"})

# Fit your models 
est1 = smf.ols("foreign_i ~ weight + length + price", data=df).fit()
est2 = smf.probit("foreign_i ~ weight + length + price", data=df).fit(disp=0)

# Make the table
mt.ETable([est1, est2], model_stats=["N","r2","pseudo_r2"], model_heads=["OLS","Probit"])
````

````{.markdown filename="Cell 10: Markdown cell"}
## Conclusion


Lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.

{{{< pagebreak >}}}

## References

::: {#refs}
:::
````

## Rendered PDF


:::
